In [90]:
import sys
from pathlib import Path
from pyprojroot import here

sys.path.append(str(here()))


In [91]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

from src.config import cfg
from src.utils import set_seed


%load_ext autoreload
%autoreload 2

The autoreload extension is already loaded. To reload it, use:
  %reload_ext autoreload


In [92]:
gseed = cfg.general.seed
set_seed(gseed)


In [93]:
train_path = Path(cfg.paths.train)
test_path = Path(cfg.paths.test)


In [94]:
df_train = pd.read_csv(train_path)
df_test = pd.read_csv(test_path)

In [95]:
df_all_data = pd.concat([df_train, df_test], axis=0).reset_index(drop=True)

df_all_data = df_all_data.drop(columns=["SalePrice"])
df_all_data

,Id,MSSubClass,MSZoning,LotFrontage,LotArea,Street,Alley,LotShape,LandContour,Utilities,...,ScreenPorch,PoolArea,PoolQC,Fence,MiscFeature,MiscVal,MoSold,YrSold,SaleType,SaleCondition
0,1,60,RL,65.0,8450,Pave,NaN,Reg,Lvl,AllPub,...,0,0,NaN,NaN,NaN,0,2,2008,WD,Normal
1,2,20,RL,80.0,9600,Pave,NaN,Reg,Lvl,AllPub,...,0,0,NaN,NaN,NaN,0,5,2007,WD,Normal
2,3,60,RL,68.0,11250,Pave,NaN,IR1,Lvl,AllPub,...,0,0,NaN,NaN,NaN,0,9,2008,WD,Normal
3,4,70,RL,60.0,9550,Pave,NaN,IR1,Lvl,AllPub,...,0,0,NaN,NaN,NaN,0,2,2006,WD,Abnorml
4,5,60,RL,84.0,14260,Pave,NaN,IR1,Lvl,AllPub,...,0,0,NaN,NaN,NaN,0,12,2008,WD,Normal
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
2914,2915,160,RM,21.0,1936,Pave,NaN,Reg,Lvl,AllPub,...,0,0,NaN,NaN,NaN,0,6,2006,WD,Normal
2915,2916,160,RM,21.0,1894,Pave,NaN,Reg,Lvl,AllPub,...,0,0,NaN,NaN,NaN,0,4,2006,WD,Abnorml
2916,2917,20,RL,160.0,20000,Pave,NaN,Reg,Lvl,AllPub,...,0,0,NaN,NaN,NaN,0,9,2006,WD,Abnorml
2917,2918,85,RL,62.0,10441,Pave,NaN,Reg,Lvl,AllPub,...,0,0,NaN,MnPrv,Shed,700,7,2006,WD,Normal


### nan imputation

In [96]:
temp = df_all_data.isna().sum()
temp[temp > 0].sort_values(ascending=False)

PoolQC          2909
MiscFeature     2814
Alley           2721
Fence           2348
MasVnrType      1766
FireplaceQu     1420
LotFrontage      486
GarageQual       159
GarageYrBlt      159
GarageCond       159
GarageFinish     159
GarageType       157
BsmtExposure      82
BsmtCond          82
BsmtQual          81
BsmtFinType2      80
BsmtFinType1      79
MasVnrArea        23
MSZoning           4
BsmtFullBath       2
Functional         2
BsmtHalfBath       2
Utilities          2
BsmtFinSF1         1
Exterior2nd        1
Exterior1st        1
Electrical         1
TotalBsmtSF        1
BsmtUnfSF          1
BsmtFinSF2         1
KitchenQual        1
GarageArea         1
GarageCars         1
SaleType           1
dtype: int64

In [97]:
df_all_data["PoolQC"] = df_all_data["PoolQC"].fillna("None")

In [98]:
df_all_data["MiscFeature"] = df_all_data["MiscFeature"].fillna("None")

In [99]:
df_all_data["Alley"] = df_all_data["Alley"].fillna("None")

In [100]:
df_all_data["Fence"] = df_all_data["Fence"].fillna("None")

In [101]:
df_all_data["FireplaceQu"] = df_all_data["FireplaceQu"].fillna("None")

In [102]:
df_all_data["MasVnrType"] = df_all_data["MasVnrType"].fillna("None")
df_all_data["MasVnrArea"] = df_all_data["MasVnrArea"].fillna(0)

In [103]:
# для бейзлайна заполнение по соседству
df_all_data["LotFrontage"] = df_all_data.groupby("Neighborhood")[
    "LotFrontage"
].transform(lambda x: x.fillna(x.median()))

In [104]:
for col in ("GarageQual", "GarageFinish", "GarageType", "GarageCond"):
    df_all_data[col] = df_all_data[col].fillna("None")

In [105]:
# для бейзлайна и так сойдет
for col in ("GarageYrBlt", "GarageArea", "GarageCars"):
    df_all_data[col] = df_all_data[col].fillna(0)

In [106]:
for col in ("BsmtQual", "BsmtCond", "BsmtExposure", "BsmtFinType1", "BsmtFinType2"):
    df_all_data[col] = df_all_data[col].fillna("None")

In [107]:
for col in (
    "BsmtFinSF1",
    "BsmtFinSF2",
    "BsmtUnfSF",
    "TotalBsmtSF",
    "BsmtFullBath",
    "BsmtHalfBath",
):
    df_all_data[col] = df_all_data[col].fillna(0)

In [108]:
df_all_data["Electrical"] = df_all_data["Electrical"].fillna(
    df_all_data["Electrical"].mode()[0]
)

In [109]:
df_all_data["MSZoning"] = df_all_data["MSZoning"].fillna(
    df_all_data["MSZoning"].mode()[0]
)

In [110]:
# вообще бесполезна, лучше убрать
df_all_data = df_all_data.drop(["Utilities"], axis=1)

In [111]:
df_all_data["Functional"] = df_all_data["Functional"].fillna("Typ")

In [112]:
df_all_data["Exterior1st"] = df_all_data["Exterior1st"].fillna(
    df_all_data["Exterior1st"].mode()[0]
)
df_all_data["Exterior2nd"] = df_all_data["Exterior2nd"].fillna(
    df_all_data["Exterior2nd"].mode()[0]
)

In [113]:
df_all_data["SaleType"] = df_all_data["SaleType"].fillna(
    df_all_data["SaleType"].mode()[0]
)

In [114]:
df_all_data["KitchenQual"] = df_all_data["KitchenQual"].fillna(
    df_all_data["KitchenQual"].mode()[0]
)

In [115]:
# пропусков больше нет
temp = df_all_data.isna().sum()
temp[temp > 0].sort_values(ascending=False)

Series([], dtype: int64)

### pipeline for no data leakages

In [116]:
from sklearn.base import BaseEstimator, TransformerMixin, OneToOneFeatureMixin

In [117]:
class BaselineTransformer(BaseEstimator, TransformerMixin):
    # признаки, где NaN = категория
    NONE_COLS = [
        "PoolQC",
        "MiscFeature",
        "Alley",
        "Fence",
        "FireplaceQu",
        "MasVnrType",
        "GarageQual",
        "GarageFinish",
        "GarageType",
        "GarageCond",
        "BsmtQual",
        "BsmtCond",
        "BsmtExposure",
        "BsmtFinType1",
        "BsmtFinType2",
    ]

    # числовые признаки, где NaN = 0
    ZERO_COLS = [
        "MasVnrArea",
        "GarageYrBlt",
        "GarageArea",
        "GarageCars",
        "BsmtFinSF1",
        "BsmtFinSF2",
        "BsmtUnfSF",
        "TotalBsmtSF",
        "BsmtFullBath",
        "BsmtHalfBath",
    ]

    # признаки, заполняемые модой (считаем на fit)
    MODE_COLS = [
        "Electrical",
        "MSZoning",
        "Exterior1st",
        "Exterior2nd",
        "SaleType",
        "KitchenQual",
    ]

    def fit(self, X_original, y=None):
        X = X_original.copy()

        self.lotfrontage_medians_ = X.groupby(by=["Neighborhood"])[
            "LotFrontage"
        ].median()

        self.lotfrontage_global_median_ = X["LotFrontage"].median()

        self.modes_ = {col: X[col].mode()[0] for col in self.MODE_COLS}

        return self

    def transform(self, X_original):
        X = X_original.copy()

        for col in self.NONE_COLS:
            X[col] = X[col].fillna("None")

        for col in self.ZERO_COLS:
            X[col] = X[col].fillna(0)

        X["LotFrontage"] = X.apply(
            lambda row: (
                self.lotfrontage_medians_.get(
                    row["Neighborhood"], self.lotfrontage_global_median_
                )
                if pd.isna(row["LotFrontage"])
                else row["LotFrontage"]
            ),
            axis=1,
        )

        for col, mode_value in self.modes_.items():
            X[col] = X[col].fillna(mode_value)

        X["Functional"] = X["Functional"].fillna("Typ")

        if "Id" in X.columns:
            X = X.drop(columns=["Id"])

        if "Utilities" in X.columns:
            X = X.drop(columns=["Utilities"])

        if "MSSubClass" in X.columns:
            X["MSSubClass"] = X["MSSubClass"].astype(str)

        return X

In [118]:
df_all_data = pd.concat([df_train, df_test], axis=0).reset_index(drop=True)

df_all_data = df_all_data.drop(columns=["SalePrice"])
df_all_data

,Id,MSSubClass,MSZoning,LotFrontage,LotArea,Street,Alley,LotShape,LandContour,Utilities,...,ScreenPorch,PoolArea,PoolQC,Fence,MiscFeature,MiscVal,MoSold,YrSold,SaleType,SaleCondition
0,1,60,RL,65.0,8450,Pave,NaN,Reg,Lvl,AllPub,...,0,0,NaN,NaN,NaN,0,2,2008,WD,Normal
1,2,20,RL,80.0,9600,Pave,NaN,Reg,Lvl,AllPub,...,0,0,NaN,NaN,NaN,0,5,2007,WD,Normal
2,3,60,RL,68.0,11250,Pave,NaN,IR1,Lvl,AllPub,...,0,0,NaN,NaN,NaN,0,9,2008,WD,Normal
3,4,70,RL,60.0,9550,Pave,NaN,IR1,Lvl,AllPub,...,0,0,NaN,NaN,NaN,0,2,2006,WD,Abnorml
4,5,60,RL,84.0,14260,Pave,NaN,IR1,Lvl,AllPub,...,0,0,NaN,NaN,NaN,0,12,2008,WD,Normal
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
2914,2915,160,RM,21.0,1936,Pave,NaN,Reg,Lvl,AllPub,...,0,0,NaN,NaN,NaN,0,6,2006,WD,Normal
2915,2916,160,RM,21.0,1894,Pave,NaN,Reg,Lvl,AllPub,...,0,0,NaN,NaN,NaN,0,4,2006,WD,Abnorml
2916,2917,20,RL,160.0,20000,Pave,NaN,Reg,Lvl,AllPub,...,0,0,NaN,NaN,NaN,0,9,2006,WD,Abnorml
2917,2918,85,RL,62.0,10441,Pave,NaN,Reg,Lvl,AllPub,...,0,0,NaN,MnPrv,Shed,700,7,2006,WD,Normal


In [119]:
transformer = BaselineTransformer()
df_all_data = transformer.fit_transform(df_all_data)
df_all_data

,MSSubClass,MSZoning,LotFrontage,LotArea,Street,Alley,LotShape,LandContour,LotConfig,LandSlope,...,ScreenPorch,PoolArea,PoolQC,Fence,MiscFeature,MiscVal,MoSold,YrSold,SaleType,SaleCondition
0,60,RL,65.0,8450,Pave,None,Reg,Lvl,Inside,Gtl,...,0,0,None,None,None,0,2,2008,WD,Normal
1,20,RL,80.0,9600,Pave,None,Reg,Lvl,FR2,Gtl,...,0,0,None,None,None,0,5,2007,WD,Normal
2,60,RL,68.0,11250,Pave,None,IR1,Lvl,Inside,Gtl,...,0,0,None,None,None,0,9,2008,WD,Normal
3,70,RL,60.0,9550,Pave,None,IR1,Lvl,Corner,Gtl,...,0,0,None,None,None,0,2,2006,WD,Abnorml
4,60,RL,84.0,14260,Pave,None,IR1,Lvl,FR2,Gtl,...,0,0,None,None,None,0,12,2008,WD,Normal
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
2914,160,RM,21.0,1936,Pave,None,Reg,Lvl,Inside,Gtl,...,0,0,None,None,None,0,6,2006,WD,Normal
2915,160,RM,21.0,1894,Pave,None,Reg,Lvl,Inside,Gtl,...,0,0,None,None,None,0,4,2006,WD,Abnorml
2916,20,RL,160.0,20000,Pave,None,Reg,Lvl,Inside,Gtl,...,0,0,None,None,None,0,9,2006,WD,Abnorml
2917,85,RL,62.0,10441,Pave,None,Reg,Lvl,Inside,Gtl,...,0,0,None,MnPrv,Shed,700,7,2006,WD,Normal


In [120]:
temp = df_all_data.isna().sum()
temp[temp > 0].sort_values(ascending=False)

Series([], dtype: int64)

In [121]:
import warnings


class MemoryOptimizer(BaseEstimator, TransformerMixin):
    def __init__(self, float16_as32=True, optimize_categories=True, cat_threshold=None):
        self.float16_as32 = float16_as32
        self.optimize_categories = optimize_categories
        self.cat_threshold = cat_threshold

    def fit(self, X, y=None):
        self._validate_input(X)

        self.categorical_cols_ = []
        self.numeric_downcast_rules_ = {}
        self.category_levels_ = {}

        for col in X.columns:
            col_type = X[col].dtype

            if self.optimize_categories and (
                col_type == "object"
                or col_type.name == "string"
                or col_type.name == "category"
            ):
                nunique = X[col].nunique()
                if self.cat_threshold is None or nunique <= self.cat_threshold:
                    self.categorical_cols_.append(col)
                    # запоминаем уровни, известные на fit, чтобы ловить unseen-категории на transform
                    self.category_levels_[col] = set(X[col].dropna().unique())
                continue

            if "int" in str(col_type):
                downcasted = pd.to_numeric(X[col], downcast="integer")
                self.numeric_downcast_rules_[col] = downcasted.dtype

            elif "float" in str(col_type):
                downcasted = pd.to_numeric(X[col], downcast="float")
                if self.float16_as32 and downcasted.dtype == np.float16:
                    self.numeric_downcast_rules_[col] = np.dtype(np.float32)
                else:
                    self.numeric_downcast_rules_[col] = downcasted.dtype

        return self

    def transform(self, X):
        self._validate_input(X)
        self._check_is_fitted()

        X_out = X.copy()
        start_mem = X_out.memory_usage(deep=True).sum() / 1024**2

        if self.optimize_categories:
            for col in self.categorical_cols_:
                if col not in X_out.columns:
                    continue

                original_na = X_out[col].isna().sum()

                X_out[col] = X_out[col].astype("object")

                known_levels = self.category_levels_[col]
                unseen_mask = X_out[col].notna() & ~X_out[col].isin(known_levels)
                n_unseen = unseen_mask.sum()

                if n_unseen > 0:
                    X_out.loc[unseen_mask, col] = np.nan
                    warnings.warn(
                        f"Column '{col}': {n_unseen} unseen categories converted to NaN on transform."
                    )

                new_na = X_out[col].isna().sum()
                if new_na > original_na and n_unseen == 0:
                    pass

        for col, target_dtype in self.numeric_downcast_rules_.items():
            if col not in X_out.columns:
                continue

            col_min, col_max = X_out[col].min(), X_out[col].max()

            if np.issubdtype(target_dtype, np.integer):
                dtype_info = np.iinfo(target_dtype)
            else:
                dtype_info = np.finfo(target_dtype)

            if col_min < dtype_info.min or col_max > dtype_info.max:
                warnings.warn(
                    f"Column '{col}': values [{col_min}, {col_max}] exceed {target_dtype} range on transform. Falling back to original dtype."
                )
                continue

            X_out[col] = X_out[col].astype(target_dtype)

        end_mem = X_out.memory_usage(deep=True).sum() / 1024**2
        percent_decrease = (
            100 * (start_mem - end_mem) / start_mem if start_mem > 0 else 0
        )
        print(
            f"Оптимизация памяти завершена: {start_mem:.2f} MB -> {end_mem:.2f} MB (-{percent_decrease:.1f}%)"
        )

        return X_out

    @staticmethod
    def _validate_input(X):
        if not isinstance(X, pd.DataFrame):
            raise TypeError("X должен быть экземпляром pandas.DataFrame")

    def _check_is_fitted(self):
        if not hasattr(self, "categorical_cols_") or not hasattr(
            self, "numeric_downcast_rules_"
        ):
            raise RuntimeError("Трансформер не обучен. Сначала вызовите fit().")

### baseline

In [122]:
from sklearn.pipeline import Pipeline

num_columns = [
    "LotFrontage",
    "LotArea",
    "OverallQual",
    "OverallCond",
    "YearBuilt",
    "YearRemodAdd",
    "MasVnrArea",
    "BsmtFinSF1",
    "BsmtFinSF2",
    "BsmtUnfSF",
    "TotalBsmtSF",
    "1stFlrSF",
    "2ndFlrSF",
    "LowQualFinSF",
    "GrLivArea",
    "BsmtFullBath",
    "BsmtHalfBath",
    "FullBath",
    "HalfBath",
    "BedroomAbvGr",
    "KitchenAbvGr",
    "TotRmsAbvGrd",
    "Fireplaces",
    "GarageYrBlt",
    "GarageCars",
    "GarageArea",
    "WoodDeckSF",
    "OpenPorchSF",
    "EnclosedPorch",
    "3SsnPorch",
    "ScreenPorch",
    "PoolArea",
    "MiscVal",
    "MoSold",
    "YrSold",
]

ordinal_columns = [
    "LotShape",  # Reg > IR1 > IR2 > IR3
    "LandSlope",  # Gtl > Mod > Sev
    "ExterQual",  # Ex > Gd > TA > Fa > Po
    "ExterCond",  # Ex > Gd > TA > Fa > Po
    "BsmtQual",  # Ex > Gd > TA > Fa > Po > None
    "BsmtCond",  # Ex > Gd > TA > Fa > Po > None
    "BsmtExposure",  # Gd > Av > Mn > No > None
    "BsmtFinType1",  # GLQ > ALQ > BLQ > Rec > LwQ > Unf > None
    "BsmtFinType2",  # GLQ > ALQ > BLQ > Rec > LwQ > Unf > None
    "HeatingQC",  # Ex > Gd > TA > Fa > Po
    "KitchenQual",  # Ex > Gd > TA > Fa > Po
    "Functional",  # Typ > Min1 > Min2 > Mod > Maj1 > Maj2 > Sev > Sal
    "FireplaceQu",  # Ex > Gd > TA > Fa > Po > None
    "GarageFinish",  # Fin > RFn > Unf > None
    "GarageQual",  # Ex > Gd > TA > Fa > Po > None
    "GarageCond",  # Ex > Gd > TA > Fa > Po > None
    "PavedDrive",  # Y > P > N
    "PoolQC",  # Ex > Gd > TA > Fa > None
    "CentralAir",  # Y > N
]

ordinal_categories = [
    ["IR3", "IR2", "IR1", "Reg"],  # LotShape
    ["Sev", "Mod", "Gtl"],  # LandSlope
    ["Po", "Fa", "TA", "Gd", "Ex"],  # ExterQual
    ["Po", "Fa", "TA", "Gd", "Ex"],  # ExterCond
    ["None", "Po", "Fa", "TA", "Gd", "Ex"],  # BsmtQual
    ["None", "Po", "Fa", "TA", "Gd", "Ex"],  # BsmtCond
    ["None", "No", "Mn", "Av", "Gd"],  # BsmtExposure
    ["None", "Unf", "LwQ", "Rec", "BLQ", "ALQ", "GLQ"],  # BsmtFinType1
    ["None", "Unf", "LwQ", "Rec", "BLQ", "ALQ", "GLQ"],  # BsmtFinType2
    ["Po", "Fa", "TA", "Gd", "Ex"],  # HeatingQC
    ["Po", "Fa", "TA", "Gd", "Ex"],  # KitchenQual
    ["Sal", "Sev", "Maj2", "Maj1", "Mod", "Min2", "Min1", "Typ"],  # Functional
    ["None", "Po", "Fa", "TA", "Gd", "Ex"],  # FireplaceQu
    ["None", "Unf", "RFn", "Fin"],  # GarageFinish
    ["None", "Po", "Fa", "TA", "Gd", "Ex"],  # GarageQual
    ["None", "Po", "Fa", "TA", "Gd", "Ex"],  # GarageCond
    ["N", "P", "Y"],  # PavedDrive
    ["None", "Fa", "TA", "Gd", "Ex"],  # PoolQC
    ["N", "Y"],  # CentralAir
]

ohe_columns = [
    "Fence",
    "Electrical",
    "MSSubClass",
    "MSZoning",
    "Street",
    "Alley",
    "LandContour",
    "LotConfig",
    "Neighborhood",
    "Condition1",
    "Condition2",
    "BldgType",
    "HouseStyle",
    "RoofStyle",
    "RoofMatl",
    "Exterior1st",
    "Exterior2nd",
    "MasVnrType",
    "Foundation",
    "Heating",
    "GarageType",
    "MiscFeature",
    "SaleType",
    "SaleCondition",
]


In [123]:
print(len(num_columns) + len(ohe_columns) + len(ordinal_columns))

78


In [124]:
from sklearn.compose import ColumnTransformer
from sklearn.preprocessing import StandardScaler, OrdinalEncoder, OneHotEncoder

### simple models

In [ ]:
linear_pipe = Pipeline(
    [
        ("baseline_transformer", BaselineTransformer()),
        ("memory_optimizer", MemoryOptimizer()),
        (
            "column_transformer",
            ColumnTransformer(
                transformers=[
                    ("num_features", StandardScaler(), num_columns),
                    (
                        "cat_features_oe",
                        OrdinalEncoder(
                            categories=ordinal_categories,
                            handle_unknown="use_encoded_value",
                            unknown_value=-1,
                        ),
                        ordinal_columns,
                    ),
                    (
                        "cat_features_ohe",
                        OneHotEncoder(drop="first", handle_unknown="ignore"),
                        ohe_columns,
                    ),
                ]
            ),
        ),
    ]
)

In [ ]:
from sklearn.base import clone
from sklearn.metrics import root_mean_squared_error

In [127]:
import traceback


def cv_result(
    model,
    X_train,
    y_train,
    y_strat,
    cv_splitter,
    preprocessor,
    name=None,
    fit_params=None,
):
    fit_params = fit_params or {}
    model_pipe = Pipeline([("preprocessor", preprocessor), ("model", model)])

    fitted_models = []
    oof_preds_sum = np.zeros(len(X_train), dtype=float)
    oof_counts = np.zeros(len(X_train), dtype=float)

    tr_scores, val_scores = [], []

    for train_idx, val_idx in cv_splitter.split(X_train, y_strat):
        X_tr, X_val = X_train.iloc[train_idx], X_train.iloc[val_idx]
        y_tr, y_val = y_train.iloc[train_idx], y_train.iloc[val_idx]

        fold_pipe = clone(model_pipe)

        fold_pipe.fit(X_tr, y_tr, **fit_params)

        fold_preds_tr = fold_pipe.predict(X_tr)

        fold_preds_val = fold_pipe.predict(X_val)

        oof_preds_sum[val_idx] += fold_preds_val
        oof_counts[val_idx] += 1

        fitted_models.append(fold_pipe)

        tr_scores.append(root_mean_squared_error(y_tr, fold_preds_tr))
        val_scores.append(root_mean_squared_error(y_val, fold_preds_val))

    oof_preds = oof_preds_sum / oof_counts

    oof_mse = root_mean_squared_error(y_train, oof_preds)
    metrics = pd.DataFrame(
        [
            {
                "model": name,
                "TRAIN_rmsle_MEAN": np.mean(tr_scores),
                "TRAIN_rmsle_STD": np.std(tr_scores),
                "VAL_rmsle_MEAN": np.mean(val_scores),
                "VAL_rmsle_STD": np.std(val_scores),
                "OOF_rmsle": oof_mse,
            }
        ]
    )

    return fitted_models, oof_preds, metrics, val_scores

In [128]:
from sklearn.linear_model import LinearRegression, Ridge
from sklearn.model_selection import RepeatedStratifiedKFold

In [129]:
X_train = df_train.drop(columns=["SalePrice"])
y_train = np.log1p(df_train["SalePrice"])  # + mse in cv loop

In [130]:
# стратификация
y_binned = pd.qcut(y_train, q=10, labels=False)

rskf = RepeatedStratifiedKFold(n_splits=10, n_repeats=5, random_state=gseed)

In [131]:
# linear regression
lr = LinearRegression()
fitted_models, oof_preds, metrics, val_scores = cv_result(
    lr, X_train, y_train, y_binned, rskf, linear_pipe, "linear_regression"
)

Оптимизация памяти завершена: 3.20 MB -> 2.93 MB (-8.4%)
Оптимизация памяти завершена: 3.20 MB -> 2.93 MB (-8.4%)
Оптимизация памяти завершена: 0.36 MB -> 0.33 MB (-8.4%)
Оптимизация памяти завершена: 3.20 MB -> 2.93 MB (-8.4%)


d:\vs_projects\fp_houses\.venv\Lib\site-packages\sklearn\preprocessing\_encoders.py:262: UserWarning: Found unknown categories in columns [15] during transform. These unknown categories will be encoded as all zeros
  warnings.warn(msg, UserWarning)


Оптимизация памяти завершена: 3.20 MB -> 2.93 MB (-8.4%)
Оптимизация памяти завершена: 0.36 MB -> 0.33 MB (-8.4%)
Оптимизация памяти завершена: 3.20 MB -> 2.93 MB (-8.4%)


d:\vs_projects\fp_houses\.venv\Lib\site-packages\sklearn\preprocessing\_encoders.py:262: UserWarning: Found unknown categories in columns [8, 10, 15] during transform. These unknown categories will be encoded as all zeros
  warnings.warn(msg, UserWarning)


Оптимизация памяти завершена: 3.20 MB -> 2.93 MB (-8.4%)
Оптимизация памяти завершена: 0.36 MB -> 0.33 MB (-8.4%)
Оптимизация памяти завершена: 3.20 MB -> 2.93 MB (-8.4%)
Оптимизация памяти завершена: 3.20 MB -> 2.93 MB (-8.4%)
Оптимизация памяти завершена: 0.36 MB -> 0.33 MB (-8.4%)
Оптимизация памяти завершена: 3.20 MB -> 2.93 MB (-8.4%)


...

d:\vs_projects\fp_houses\.venv\Lib\site-packages\sklearn\preprocessing\_encoders.py:262: UserWarning: Found unknown categories in columns [15] during transform. These unknown categories will be encoded as all zeros
  warnings.warn(msg, UserWarning)


Оптимизация памяти завершена: 3.20 MB -> 2.93 MB (-8.4%)
Оптимизация памяти завершена: 0.36 MB -> 0.33 MB (-8.4%)
Оптимизация памяти завершена: 3.20 MB -> 2.93 MB (-8.4%)


d:\vs_projects\fp_houses\.venv\Lib\site-packages\sklearn\preprocessing\_encoders.py:262: UserWarning: Found unknown categories in columns [1, 10, 14, 16] during transform. These unknown categories will be encoded as all zeros
  warnings.warn(msg, UserWarning)


Оптимизация памяти завершена: 3.20 MB -> 2.93 MB (-8.4%)
Оптимизация памяти завершена: 0.36 MB -> 0.33 MB (-8.4%)


d:\vs_projects\fp_houses\.venv\Lib\site-packages\sklearn\preprocessing\_encoders.py:262: UserWarning: Found unknown categories in columns [21] during transform. These unknown categories will be encoded as all zeros
  warnings.warn(msg, UserWarning)


In [132]:
metrics

,model,TRAIN_rmsle_MEAN,TRAIN_rmsle_STD,VAL_rmsle_MEAN,VAL_rmsle_STD,OOF_rmsle
0,linear_regression,0.095523,0.002172,0.18553,0.060848,0.193909


In [133]:
# ridge regression
rr = Ridge(alpha=1)
fitted_models, oof_preds, metrics, val_scores = cv_result(
    rr, X_train, y_train, y_binned, rskf, linear_pipe, "ridge_regression"
)

Оптимизация памяти завершена: 3.20 MB -> 2.93 MB (-8.4%)
Оптимизация памяти завершена: 3.20 MB -> 2.93 MB (-8.4%)
Оптимизация памяти завершена: 0.36 MB -> 0.33 MB (-8.4%)
Оптимизация памяти завершена: 3.20 MB -> 2.93 MB (-8.4%)
Оптимизация памяти завершена: 3.20 MB -> 2.93 MB (-8.4%)
Оптимизация памяти завершена: 0.36 MB -> 0.33 MB (-8.4%)


d:\vs_projects\fp_houses\.venv\Lib\site-packages\sklearn\preprocessing\_encoders.py:262: UserWarning: Found unknown categories in columns [15] during transform. These unknown categories will be encoded as all zeros
  warnings.warn(msg, UserWarning)
d:\vs_projects\fp_houses\.venv\Lib\site-packages\sklearn\preprocessing\_encoders.py:262: UserWarning: Found unknown categories in columns [8, 10, 15] during transform. These unknown categories will be encoded as all zeros
  warnings.warn(msg, UserWarning)


Оптимизация памяти завершена: 3.20 MB -> 2.93 MB (-8.4%)
Оптимизация памяти завершена: 3.20 MB -> 2.93 MB (-8.4%)
Оптимизация памяти завершена: 0.36 MB -> 0.33 MB (-8.4%)
Оптимизация памяти завершена: 3.20 MB -> 2.93 MB (-8.4%)
Оптимизация памяти завершена: 3.20 MB -> 2.93 MB (-8.4%)
Оптимизация памяти завершена: 0.36 MB -> 0.33 MB (-8.4%)
Оптимизация памяти завершена: 3.20 MB -> 2.93 MB (-8.4%)
Оптимизация памяти завершена: 3.20 MB -> 2.93 MB (-8.4%)
Оптимизация памяти завершена: 0.36 MB -> 0.33 MB (-8.4%)
Оптимизация памяти завершена: 3.20 MB -> 2.93 MB (-8.4%)


d:\vs_projects\fp_houses\.venv\Lib\site-packages\sklearn\preprocessing\_encoders.py:262: UserWarning: Found unknown categories in columns [1, 14] during transform. These unknown categories will be encoded as all zeros
  warnings.warn(msg, UserWarning)
d:\vs_projects\fp_houses\.venv\Lib\site-packages\sklearn\preprocessing\_encoders.py:262: UserWarning: Found unknown categories in columns [14, 15, 16] during transform. These unknown categories will be encoded as all zeros
  warnings.warn(msg, UserWarning)


Оптимизация памяти завершена: 3.20 MB -> 2.93 MB (-8.4%)
Оптимизация памяти завершена: 0.36 MB -> 0.33 MB (-8.4%)
Оптимизация памяти завершена: 3.20 MB -> 2.93 MB (-8.4%)
Оптимизация памяти завершена: 3.20 MB -> 2.93 MB (-8.4%)
Оптимизация памяти завершена: 0.36 MB -> 0.33 MB (-8.4%)
Оптимизация памяти завершена: 3.20 MB -> 2.93 MB (-8.4%)
Оптимизация памяти завершена: 3.20 MB -> 2.93 MB (-8.4%)
Оптимизация памяти завершена: 0.36 MB -> 0.33 MB (-8.4%)
Оптимизация памяти завершена: 3.20 MB -> 2.93 MB (-8.4%)


...

d:\vs_projects\fp_houses\.venv\Lib\site-packages\sklearn\preprocessing\_encoders.py:262: UserWarning: Found unknown categories in columns [15, 16] during transform. These unknown categories will be encoded as all zeros
  warnings.warn(msg, UserWarning)
d:\vs_projects\fp_houses\.venv\Lib\site-packages\sklearn\preprocessing\_encoders.py:262: UserWarning: Found unknown categories in columns [14, 15] during transform. These unknown categories will be encoded as all zeros
  warnings.warn(msg, UserWarning)


Оптимизация памяти завершена: 3.20 MB -> 2.93 MB (-8.4%)
Оптимизация памяти завершена: 3.20 MB -> 2.93 MB (-8.4%)
Оптимизация памяти завершена: 0.36 MB -> 0.33 MB (-8.4%)
Оптимизация памяти завершена: 3.20 MB -> 2.93 MB (-8.4%)
Оптимизация памяти завершена: 3.20 MB -> 2.93 MB (-8.4%)
Оптимизация памяти завершена: 0.36 MB -> 0.33 MB (-8.4%)


d:\vs_projects\fp_houses\.venv\Lib\site-packages\sklearn\preprocessing\_encoders.py:262: UserWarning: Found unknown categories in columns [10] during transform. These unknown categories will be encoded as all zeros
  warnings.warn(msg, UserWarning)
d:\vs_projects\fp_houses\.venv\Lib\site-packages\sklearn\preprocessing\_encoders.py:262: UserWarning: Found unknown categories in columns [15] during transform. These unknown categories will be encoded as all zeros
  warnings.warn(msg, UserWarning)


Оптимизация памяти завершена: 3.20 MB -> 2.93 MB (-8.4%)
Оптимизация памяти завершена: 3.20 MB -> 2.93 MB (-8.4%)
Оптимизация памяти завершена: 0.36 MB -> 0.33 MB (-8.4%)
Оптимизация памяти завершена: 3.20 MB -> 2.93 MB (-8.4%)
Оптимизация памяти завершена: 3.20 MB -> 2.93 MB (-8.4%)
Оптимизация памяти завершена: 0.36 MB -> 0.33 MB (-8.4%)


d:\vs_projects\fp_houses\.venv\Lib\site-packages\sklearn\preprocessing\_encoders.py:262: UserWarning: Found unknown categories in columns [1, 10, 14, 16] during transform. These unknown categories will be encoded as all zeros
  warnings.warn(msg, UserWarning)
d:\vs_projects\fp_houses\.venv\Lib\site-packages\sklearn\preprocessing\_encoders.py:262: UserWarning: Found unknown categories in columns [21] during transform. These unknown categories will be encoded as all zeros
  warnings.warn(msg, UserWarning)


In [134]:
metrics

,model,TRAIN_rmsle_MEAN,TRAIN_rmsle_STD,VAL_rmsle_MEAN,VAL_rmsle_STD,OOF_rmsle
0,ridge_regression,0.106203,0.003154,0.14092,0.039436,0.145016


In [135]:
from sklearn.svm import SVR

In [136]:
# cvr
svr = SVR()
fitted_models, oof_preds, metrics, val_scores = cv_result(
    svr, X_train, y_train, y_binned, rskf, linear_pipe, "cvr"
)

Оптимизация памяти завершена: 3.20 MB -> 2.93 MB (-8.4%)
Оптимизация памяти завершена: 3.20 MB -> 2.93 MB (-8.4%)
Оптимизация памяти завершена: 0.36 MB -> 0.33 MB (-8.4%)
Оптимизация памяти завершена: 3.20 MB -> 2.93 MB (-8.4%)


d:\vs_projects\fp_houses\.venv\Lib\site-packages\sklearn\preprocessing\_encoders.py:262: UserWarning: Found unknown categories in columns [15] during transform. These unknown categories will be encoded as all zeros
  warnings.warn(msg, UserWarning)
d:\vs_projects\fp_houses\.venv\Lib\site-packages\sklearn\preprocessing\_encoders.py:262: UserWarning: Found unknown categories in columns [8, 10, 15] during transform. These unknown categories will be encoded as all zeros
  warnings.warn(msg, UserWarning)


Оптимизация памяти завершена: 3.20 MB -> 2.93 MB (-8.4%)
Оптимизация памяти завершена: 0.36 MB -> 0.33 MB (-8.4%)
Оптимизация памяти завершена: 3.20 MB -> 2.93 MB (-8.4%)
Оптимизация памяти завершена: 3.20 MB -> 2.93 MB (-8.4%)
Оптимизация памяти завершена: 0.36 MB -> 0.33 MB (-8.4%)
Оптимизация памяти завершена: 3.20 MB -> 2.93 MB (-8.4%)
Оптимизация памяти завершена: 3.20 MB -> 2.93 MB (-8.4%)
Оптимизация памяти завершена: 0.36 MB -> 0.33 MB (-8.4%)
Оптимизация памяти завершена: 3.20 MB -> 2.93 MB (-8.4%)
Оптимизация памяти завершена: 3.20 MB -> 2.93 MB (-8.4%)


d:\vs_projects\fp_houses\.venv\Lib\site-packages\sklearn\preprocessing\_encoders.py:262: UserWarning: Found unknown categories in columns [1, 14] during transform. These unknown categories will be encoded as all zeros
  warnings.warn(msg, UserWarning)
d:\vs_projects\fp_houses\.venv\Lib\site-packages\sklearn\preprocessing\_encoders.py:262: UserWarning: Found unknown categories in columns [14, 15, 16] during transform. These unknown categories will be encoded as all zeros
  warnings.warn(msg, UserWarning)


Оптимизация памяти завершена: 0.36 MB -> 0.33 MB (-8.4%)
Оптимизация памяти завершена: 3.20 MB -> 2.93 MB (-8.4%)
Оптимизация памяти завершена: 3.20 MB -> 2.93 MB (-8.4%)
Оптимизация памяти завершена: 0.36 MB -> 0.33 MB (-8.4%)
Оптимизация памяти завершена: 3.20 MB -> 2.93 MB (-8.4%)
Оптимизация памяти завершена: 3.20 MB -> 2.93 MB (-8.4%)
Оптимизация памяти завершена: 0.36 MB -> 0.33 MB (-8.4%)
Оптимизация памяти завершена: 3.20 MB -> 2.93 MB (-8.4%)
Оптимизация памяти завершена: 3.20 MB -> 2.93 MB (-8.4%)


...

d:\vs_projects\fp_houses\.venv\Lib\site-packages\sklearn\preprocessing\_encoders.py:262: UserWarning: Found unknown categories in columns [15] during transform. These unknown categories will be encoded as all zeros
  warnings.warn(msg, UserWarning)


Оптимизация памяти завершена: 3.20 MB -> 2.93 MB (-8.4%)
Оптимизация памяти завершена: 0.36 MB -> 0.33 MB (-8.4%)
Оптимизация памяти завершена: 3.20 MB -> 2.93 MB (-8.4%)


d:\vs_projects\fp_houses\.venv\Lib\site-packages\sklearn\preprocessing\_encoders.py:262: UserWarning: Found unknown categories in columns [1, 10, 14, 16] during transform. These unknown categories will be encoded as all zeros
  warnings.warn(msg, UserWarning)


Оптимизация памяти завершена: 3.20 MB -> 2.93 MB (-8.4%)
Оптимизация памяти завершена: 0.36 MB -> 0.33 MB (-8.4%)


d:\vs_projects\fp_houses\.venv\Lib\site-packages\sklearn\preprocessing\_encoders.py:262: UserWarning: Found unknown categories in columns [21] during transform. These unknown categories will be encoded as all zeros
  warnings.warn(msg, UserWarning)


In [137]:
metrics

,model,TRAIN_rmsle_MEAN,TRAIN_rmsle_STD,VAL_rmsle_MEAN,VAL_rmsle_STD,OOF_rmsle
0,cvr,0.093238,0.001617,0.128043,0.018641,0.128616


In [138]:
from sklearn.neighbors import KNeighborsRegressor

In [139]:
# knn
knn = KNeighborsRegressor()
fitted_models, oof_preds, metrics, val_scores = cv_result(
    knn, X_train, y_train, y_binned, rskf, linear_pipe, "knn_regressor"
)

Оптимизация памяти завершена: 3.20 MB -> 2.93 MB (-8.4%)
Оптимизация памяти завершена: 3.20 MB -> 2.93 MB (-8.4%)
Оптимизация памяти завершена: 0.36 MB -> 0.33 MB (-8.4%)
Оптимизация памяти завершена: 3.20 MB -> 2.93 MB (-8.4%)


d:\vs_projects\fp_houses\.venv\Lib\site-packages\sklearn\preprocessing\_encoders.py:262: UserWarning: Found unknown categories in columns [15] during transform. These unknown categories will be encoded as all zeros
  warnings.warn(msg, UserWarning)
d:\vs_projects\fp_houses\.venv\Lib\site-packages\sklearn\preprocessing\_encoders.py:262: UserWarning: Found unknown categories in columns [8, 10, 15] during transform. These unknown categories will be encoded as all zeros
  warnings.warn(msg, UserWarning)


Оптимизация памяти завершена: 3.20 MB -> 2.93 MB (-8.4%)
Оптимизация памяти завершена: 0.36 MB -> 0.33 MB (-8.4%)
Оптимизация памяти завершена: 3.20 MB -> 2.93 MB (-8.4%)
Оптимизация памяти завершена: 3.20 MB -> 2.93 MB (-8.4%)
Оптимизация памяти завершена: 0.36 MB -> 0.33 MB (-8.4%)
Оптимизация памяти завершена: 3.20 MB -> 2.93 MB (-8.4%)
Оптимизация памяти завершена: 3.20 MB -> 2.93 MB (-8.4%)
Оптимизация памяти завершена: 0.36 MB -> 0.33 MB (-8.4%)


d:\vs_projects\fp_houses\.venv\Lib\site-packages\sklearn\preprocessing\_encoders.py:262: UserWarning: Found unknown categories in columns [1, 14] during transform. These unknown categories will be encoded as all zeros
  warnings.warn(msg, UserWarning)
d:\vs_projects\fp_houses\.venv\Lib\site-packages\sklearn\preprocessing\_encoders.py:262: UserWarning: Found unknown categories in columns [14, 15, 16] during transform. These unknown categories will be encoded as all zeros
  warnings.warn(msg, UserWarning)


Оптимизация памяти завершена: 3.20 MB -> 2.93 MB (-8.4%)
Оптимизация памяти завершена: 3.20 MB -> 2.93 MB (-8.4%)
Оптимизация памяти завершена: 0.36 MB -> 0.33 MB (-8.4%)
Оптимизация памяти завершена: 3.20 MB -> 2.93 MB (-8.4%)
Оптимизация памяти завершена: 3.20 MB -> 2.93 MB (-8.4%)
Оптимизация памяти завершена: 0.36 MB -> 0.33 MB (-8.4%)
Оптимизация памяти завершена: 3.20 MB -> 2.93 MB (-8.4%)
Оптимизация памяти завершена: 3.20 MB -> 2.93 MB (-8.4%)
Оптимизация памяти завершена: 0.36 MB -> 0.33 MB (-8.4%)


...

d:\vs_projects\fp_houses\.venv\Lib\site-packages\sklearn\preprocessing\_encoders.py:262: UserWarning: Found unknown categories in columns [15, 16] during transform. These unknown categories will be encoded as all zeros
  warnings.warn(msg, UserWarning)
d:\vs_projects\fp_houses\.venv\Lib\site-packages\sklearn\preprocessing\_encoders.py:262: UserWarning: Found unknown categories in columns [14, 15] during transform. These unknown categories will be encoded as all zeros
  warnings.warn(msg, UserWarning)


Оптимизация памяти завершена: 0.36 MB -> 0.33 MB (-8.4%)
Оптимизация памяти завершена: 3.20 MB -> 2.93 MB (-8.4%)
Оптимизация памяти завершена: 3.20 MB -> 2.93 MB (-8.4%)
Оптимизация памяти завершена: 0.36 MB -> 0.33 MB (-8.4%)
Оптимизация памяти завершена: 3.20 MB -> 2.93 MB (-8.4%)
Оптимизация памяти завершена: 3.20 MB -> 2.93 MB (-8.4%)
Оптимизация памяти завершена: 0.36 MB -> 0.33 MB (-8.4%)


d:\vs_projects\fp_houses\.venv\Lib\site-packages\sklearn\preprocessing\_encoders.py:262: UserWarning: Found unknown categories in columns [10] during transform. These unknown categories will be encoded as all zeros
  warnings.warn(msg, UserWarning)
d:\vs_projects\fp_houses\.venv\Lib\site-packages\sklearn\preprocessing\_encoders.py:262: UserWarning: Found unknown categories in columns [15] during transform. These unknown categories will be encoded as all zeros
  warnings.warn(msg, UserWarning)


Оптимизация памяти завершена: 3.20 MB -> 2.93 MB (-8.4%)
Оптимизация памяти завершена: 3.20 MB -> 2.93 MB (-8.4%)
Оптимизация памяти завершена: 0.36 MB -> 0.33 MB (-8.4%)
Оптимизация памяти завершена: 3.20 MB -> 2.93 MB (-8.4%)
Оптимизация памяти завершена: 3.20 MB -> 2.93 MB (-8.4%)
Оптимизация памяти завершена: 0.36 MB -> 0.33 MB (-8.4%)


d:\vs_projects\fp_houses\.venv\Lib\site-packages\sklearn\preprocessing\_encoders.py:262: UserWarning: Found unknown categories in columns [1, 10, 14, 16] during transform. These unknown categories will be encoded as all zeros
  warnings.warn(msg, UserWarning)
d:\vs_projects\fp_houses\.venv\Lib\site-packages\sklearn\preprocessing\_encoders.py:262: UserWarning: Found unknown categories in columns [21] during transform. These unknown categories will be encoded as all zeros
  warnings.warn(msg, UserWarning)


In [140]:
metrics

,model,TRAIN_rmsle_MEAN,TRAIN_rmsle_STD,VAL_rmsle_MEAN,VAL_rmsle_STD,OOF_rmsle
0,knn_regressor,0.141765,0.001968,0.175327,0.018612,0.174046


### tree models

In [141]:
tree_pipe_with_ohe = Pipeline(
    [
        ("baseline_transformer", BaselineTransformer()),
        ("memory_optimizer", MemoryOptimizer()),
        (
            "column_transformer",
            ColumnTransformer(
                transformers=[
                    ("num_features", "passthrough", num_columns),
                    (
                        "cat_features_oe",
                        OrdinalEncoder(
                            categories=ordinal_categories,
                            handle_unknown="use_encoded_value",
                            unknown_value=-1,
                        ),
                        ordinal_columns,
                    ),
                    (
                        "cat_features_ohe",
                        OneHotEncoder(drop="first", handle_unknown="ignore"),
                        ohe_columns,
                    ),
                ]
            ),
        ),
    ]
)

In [142]:
from sklearn.ensemble import RandomForestRegressor

In [143]:
rfr = RandomForestRegressor(random_state=gseed)
fitted_models, oof_preds, metrics, val_scores = cv_result(
    rfr, X_train, y_train, y_binned, rskf, tree_pipe_with_ohe, "rfr_regressor"
)

Оптимизация памяти завершена: 3.20 MB -> 2.93 MB (-8.4%)
Оптимизация памяти завершена: 3.20 MB -> 2.93 MB (-8.4%)
Оптимизация памяти завершена: 0.36 MB -> 0.33 MB (-8.4%)
Оптимизация памяти завершена: 3.20 MB -> 2.93 MB (-8.4%)


d:\vs_projects\fp_houses\.venv\Lib\site-packages\sklearn\preprocessing\_encoders.py:262: UserWarning: Found unknown categories in columns [15] during transform. These unknown categories will be encoded as all zeros
  warnings.warn(msg, UserWarning)


Оптимизация памяти завершена: 3.20 MB -> 2.93 MB (-8.4%)
Оптимизация памяти завершена: 0.36 MB -> 0.33 MB (-8.4%)
Оптимизация памяти завершена: 3.20 MB -> 2.93 MB (-8.4%)


d:\vs_projects\fp_houses\.venv\Lib\site-packages\sklearn\preprocessing\_encoders.py:262: UserWarning: Found unknown categories in columns [8, 10, 15] during transform. These unknown categories will be encoded as all zeros
  warnings.warn(msg, UserWarning)


Оптимизация памяти завершена: 3.20 MB -> 2.93 MB (-8.4%)
Оптимизация памяти завершена: 0.36 MB -> 0.33 MB (-8.4%)
Оптимизация памяти завершена: 3.20 MB -> 2.93 MB (-8.4%)
Оптимизация памяти завершена: 3.20 MB -> 2.93 MB (-8.4%)
Оптимизация памяти завершена: 0.36 MB -> 0.33 MB (-8.4%)
Оптимизация памяти завершена: 3.20 MB -> 2.93 MB (-8.4%)


...

d:\vs_projects\fp_houses\.venv\Lib\site-packages\sklearn\preprocessing\_encoders.py:262: UserWarning: Found unknown categories in columns [15] during transform. These unknown categories will be encoded as all zeros
  warnings.warn(msg, UserWarning)


Оптимизация памяти завершена: 3.20 MB -> 2.93 MB (-8.4%)
Оптимизация памяти завершена: 0.36 MB -> 0.33 MB (-8.4%)
Оптимизация памяти завершена: 3.20 MB -> 2.93 MB (-8.4%)


d:\vs_projects\fp_houses\.venv\Lib\site-packages\sklearn\preprocessing\_encoders.py:262: UserWarning: Found unknown categories in columns [1, 10, 14, 16] during transform. These unknown categories will be encoded as all zeros
  warnings.warn(msg, UserWarning)


Оптимизация памяти завершена: 3.20 MB -> 2.93 MB (-8.4%)
Оптимизация памяти завершена: 0.36 MB -> 0.33 MB (-8.4%)


d:\vs_projects\fp_houses\.venv\Lib\site-packages\sklearn\preprocessing\_encoders.py:262: UserWarning: Found unknown categories in columns [21] during transform. These unknown categories will be encoded as all zeros
  warnings.warn(msg, UserWarning)


In [144]:
metrics

,model,TRAIN_rmsle_MEAN,TRAIN_rmsle_STD,VAL_rmsle_MEAN,VAL_rmsle_STD,OOF_rmsle
0,rfr_regressor,0.053987,0.000942,0.142462,0.017317,0.141823


In [145]:
from lightgbm import LGBMRegressor

In [146]:
lgbm = LGBMRegressor(random_state=gseed)
fitted_models, oof_preds, metrics, val_scores = cv_result(
    lgbm, X_train, y_train, y_binned, rskf, tree_pipe_with_ohe, "lgbm_regressor"
)

Оптимизация памяти завершена: 3.20 MB -> 2.93 MB (-8.4%)
[LightGBM] [Info] Auto-choosing col-wise multi-threading, the overhead of testing was 0.001379 seconds.
You can set `force_col_wise=true` to remove the overhead.
[LightGBM] [Info] Total Bins 3321
[LightGBM] [Info] Number of data points in the train set: 1314, number of used features: 144
[LightGBM] [Info] Start training from score 12.024176
Оптимизация памяти завершена: 3.20 MB -> 2.93 MB (-8.4%)
Оптимизация памяти завершена: 0.36 MB -> 0.33 MB (-8.4%)
Оптимизация памяти завершена: 3.20 MB -> 2.93 MB (-8.4%)
[LightGBM] [Info] Auto-choosing col-wise multi-threading, the overhead of testing was 0.000936 seconds.
You can set `force_col_wise=true` to remove the overhead.
[LightGBM] [Info] Total Bins 3316
[LightGBM] [Info] Number of data points in the train set: 1314, number of used features: 145
[LightGBM] [Info] Start training from score 12.022495
Оптимизация памяти завершена: 3.20 MB -> 2.93 MB (-8.4%)


d:\vs_projects\fp_houses\.venv\Lib\site-packages\sklearn\preprocessing\_encoders.py:262: UserWarning: Found unknown categories in columns [15] during transform. These unknown categories will be encoded as all zeros
  warnings.warn(msg, UserWarning)


Оптимизация памяти завершена: 0.36 MB -> 0.33 MB (-8.4%)
Оптимизация памяти завершена: 3.20 MB -> 2.93 MB (-8.4%)
[LightGBM] [Info] Auto-choosing col-wise multi-threading, the overhead of testing was 0.000848 seconds.
You can set `force_col_wise=true` to remove the overhead.
[LightGBM] [Info] Total Bins 3324
[LightGBM] [Info] Number of data points in the train set: 1314, number of used features: 145
[LightGBM] [Info] Start training from score 12.024325
Оптимизация памяти завершена: 3.20 MB -> 2.93 MB (-8.4%)


d:\vs_projects\fp_houses\.venv\Lib\site-packages\sklearn\preprocessing\_encoders.py:262: UserWarning: Found unknown categories in columns [8, 10, 15] during transform. These unknown categories will be encoded as all zeros
  warnings.warn(msg, UserWarning)


Оптимизация памяти завершена: 0.36 MB -> 0.33 MB (-8.4%)
Оптимизация памяти завершена: 3.20 MB -> 2.93 MB (-8.4%)
[LightGBM] [Info] Auto-choosing col-wise multi-threading, the overhead of testing was 0.001082 seconds.
You can set `force_col_wise=true` to remove the overhead.
[LightGBM] [Info] Total Bins 3318
[LightGBM] [Info] Number of data points in the train set: 1314, number of used features: 146
[LightGBM] [Info] Start training from score 12.024615
Оптимизация памяти завершена: 3.20 MB -> 2.93 MB (-8.4%)
Оптимизация памяти завершена: 0.36 MB -> 0.33 MB (-8.4%)
Оптимизация памяти завершена: 3.20 MB -> 2.93 MB (-8.4%)
[LightGBM] [Info] Auto-choosing col-wise multi-threading, the overhead of testing was 0.000870 seconds.
You can set `force_col_wise=true` to remove the overhead.
[LightGBM] [Info] Total Bins 3329
[LightGBM] [Info] Number of data points in the train set: 1314, number of used features: 145
[LightGBM] [Info] Start training from score 12.025350
Оптимизация памяти завершена:

...

d:\vs_projects\fp_houses\.venv\Lib\site-packages\sklearn\preprocessing\_encoders.py:262: UserWarning: Found unknown categories in columns [15] during transform. These unknown categories will be encoded as all zeros
  warnings.warn(msg, UserWarning)


Оптимизация памяти завершена: 3.20 MB -> 2.93 MB (-8.4%)
Оптимизация памяти завершена: 0.36 MB -> 0.33 MB (-8.4%)
Оптимизация памяти завершена: 3.20 MB -> 2.93 MB (-8.4%)
[LightGBM] [Info] Auto-choosing col-wise multi-threading, the overhead of testing was 0.000940 seconds.
You can set `force_col_wise=true` to remove the overhead.
[LightGBM] [Info] Total Bins 3327
[LightGBM] [Info] Number of data points in the train set: 1314, number of used features: 147
[LightGBM] [Info] Start training from score 12.024486


d:\vs_projects\fp_houses\.venv\Lib\site-packages\sklearn\preprocessing\_encoders.py:262: UserWarning: Found unknown categories in columns [1, 10, 14, 16] during transform. These unknown categories will be encoded as all zeros
  warnings.warn(msg, UserWarning)


Оптимизация памяти завершена: 3.20 MB -> 2.93 MB (-8.4%)
Оптимизация памяти завершена: 0.36 MB -> 0.33 MB (-8.4%)


d:\vs_projects\fp_houses\.venv\Lib\site-packages\sklearn\preprocessing\_encoders.py:262: UserWarning: Found unknown categories in columns [21] during transform. These unknown categories will be encoded as all zeros
  warnings.warn(msg, UserWarning)


In [147]:
metrics

,model,TRAIN_rmsle_MEAN,TRAIN_rmsle_STD,VAL_rmsle_MEAN,VAL_rmsle_STD,OOF_rmsle
0,lgbm_regressor,0.044258,0.001054,0.129718,0.016017,0.127076


In [148]:
class ToCategory(OneToOneFeatureMixin, BaseEstimator, TransformerMixin):
    def __init__(self, columns):
        self.columns = columns

    def fit(self, X, y=None):
        self.n_features_in_ = X.shape[1]
        self.feature_names_in_ = np.asarray(X.columns)
        self.categories_ = {
            col: pd.Index(X[col].astype("category").cat.categories)
            for col in self.columns
        }
        return self

    def transform(self, X):
        X_copy = X.copy()
        for col in self.columns:
            X_copy[col] = pd.Categorical(X_copy[col], categories=self.categories_[col])
        return X_copy

### encoders comparison

In [149]:
tree_pipe_enc = Pipeline(
    [
        ("baseline_transformer", BaselineTransformer()),
        ("memory_optimizer", MemoryOptimizer()),
        (
            "column_transformer",
            ColumnTransformer(
                transformers=[
                    ("num_features", "passthrough", num_columns),
                    (
                        "cat_features_oe",
                        # OrdinalEncoder(
                        #     categories=ordinal_categories,
                        #     handle_unknown="use_encoded_value",
                        #     unknown_value=-1,
                        # ),
                        "passthrough",
                        ordinal_columns,
                    ),
                    (
                        "cat_features_ohe",
                        "passthrough",
                        ohe_columns,
                    ),
                ],
                verbose_feature_names_out=False,
            ).set_output(transform="pandas"),
        ),
        ("to_category", ToCategory(ordinal_columns + ohe_columns)),
    ]
)

In [150]:
lgbm = LGBMRegressor(random_state=gseed)
fitted_models, oof_preds, metrics, val_scores = cv_result(
    lgbm,
    X_train,
    y_train,
    y_binned,
    rskf,
    tree_pipe_enc,
    "lgbm_regressor_auto_encoder",
)

Оптимизация памяти завершена: 3.20 MB -> 2.93 MB (-8.4%)
[LightGBM] [Info] Auto-choosing col-wise multi-threading, the overhead of testing was 0.001258 seconds.
You can set `force_col_wise=true` to remove the overhead.
[LightGBM] [Info] Total Bins 3318
[LightGBM] [Info] Number of data points in the train set: 1314, number of used features: 76
[LightGBM] [Info] Start training from score 12.024176
Оптимизация памяти завершена: 3.20 MB -> 2.93 MB (-8.4%)
Оптимизация памяти завершена: 0.36 MB -> 0.33 MB (-8.4%)
Оптимизация памяти завершена: 3.20 MB -> 2.93 MB (-8.4%)
[LightGBM] [Info] Auto-choosing col-wise multi-threading, the overhead of testing was 0.000539 seconds.
You can set `force_col_wise=true` to remove the overhead.
[LightGBM] [Info] Total Bins 3311
[LightGBM] [Info] Number of data points in the train set: 1314, number of used features: 76
[LightGBM] [Info] Start training from score 12.022495


C:\Users\kochn\AppData\Local\Temp\ipykernel_27260\1431418370.py:17: Pandas4Warning: Constructing a Categorical with a dtype and values containing non-null entries not in that dtype's categories is deprecated and will raise in a future version.
  X_copy[col] = pd.Categorical(X_copy[col], categories=self.categories_[col])


Оптимизация памяти завершена: 3.20 MB -> 2.93 MB (-8.4%)
Оптимизация памяти завершена: 0.36 MB -> 0.33 MB (-8.4%)
Оптимизация памяти завершена: 3.20 MB -> 2.93 MB (-8.4%)
[LightGBM] [Info] Auto-choosing col-wise multi-threading, the overhead of testing was 0.000771 seconds.
You can set `force_col_wise=true` to remove the overhead.
[LightGBM] [Info] Total Bins 3320
[LightGBM] [Info] Number of data points in the train set: 1314, number of used features: 76
[LightGBM] [Info] Start training from score 12.024325


C:\Users\kochn\AppData\Local\Temp\ipykernel_27260\1431418370.py:17: Pandas4Warning: Constructing a Categorical with a dtype and values containing non-null entries not in that dtype's categories is deprecated and will raise in a future version.
  X_copy[col] = pd.Categorical(X_copy[col], categories=self.categories_[col])


Оптимизация памяти завершена: 3.20 MB -> 2.93 MB (-8.4%)
Оптимизация памяти завершена: 0.36 MB -> 0.33 MB (-8.4%)
Оптимизация памяти завершена: 3.20 MB -> 2.93 MB (-8.4%)
[LightGBM] [Info] Auto-choosing col-wise multi-threading, the overhead of testing was 0.000617 seconds.
You can set `force_col_wise=true` to remove the overhead.
[LightGBM] [Info] Total Bins 3312
[LightGBM] [Info] Number of data points in the train set: 1314, number of used features: 76
[LightGBM] [Info] Start training from score 12.024615


...

C:\Users\kochn\AppData\Local\Temp\ipykernel_27260\1431418370.py:17: Pandas4Warning: Constructing a Categorical with a dtype and values containing non-null entries not in that dtype's categories is deprecated and will raise in a future version.
  X_copy[col] = pd.Categorical(X_copy[col], categories=self.categories_[col])


Оптимизация памяти завершена: 3.20 MB -> 2.93 MB (-8.4%)
Оптимизация памяти завершена: 0.36 MB -> 0.33 MB (-8.4%)
Оптимизация памяти завершена: 3.20 MB -> 2.93 MB (-8.4%)
[LightGBM] [Info] Auto-choosing col-wise multi-threading, the overhead of testing was 0.000708 seconds.
You can set `force_col_wise=true` to remove the overhead.
[LightGBM] [Info] Total Bins 3319
[LightGBM] [Info] Number of data points in the train set: 1314, number of used features: 76
[LightGBM] [Info] Start training from score 12.024486


C:\Users\kochn\AppData\Local\Temp\ipykernel_27260\1431418370.py:17: Pandas4Warning: Constructing a Categorical with a dtype and values containing non-null entries not in that dtype's categories is deprecated and will raise in a future version.
  X_copy[col] = pd.Categorical(X_copy[col], categories=self.categories_[col])


Оптимизация памяти завершена: 3.20 MB -> 2.93 MB (-8.4%)
Оптимизация памяти завершена: 0.36 MB -> 0.33 MB (-8.4%)


C:\Users\kochn\AppData\Local\Temp\ipykernel_27260\1431418370.py:17: Pandas4Warning: Constructing a Categorical with a dtype and values containing non-null entries not in that dtype's categories is deprecated and will raise in a future version.
  X_copy[col] = pd.Categorical(X_copy[col], categories=self.categories_[col])


In [151]:
metrics

,model,TRAIN_rmsle_MEAN,TRAIN_rmsle_STD,VAL_rmsle_MEAN,VAL_rmsle_STD,OOF_rmsle
0,lgbm_regressor_auto_encoder,0.040153,0.000801,0.126993,0.014737,0.124109


In [155]:
tree_pipe_enc = Pipeline(
    [
        ("baseline_transformer", BaselineTransformer()),
        ("memory_optimizer", MemoryOptimizer()),
        (
            "column_transformer",
            ColumnTransformer(
                transformers=[
                    ("num_features", "passthrough", num_columns),
                    (
                        "cat_features_oe",
                        OrdinalEncoder(
                            categories=ordinal_categories,
                            handle_unknown="use_encoded_value",
                            unknown_value=-1,
                        ),
                        ordinal_columns,
                    ),
                    (
                        "cat_features_ohe",
                        "passthrough",
                        ohe_columns,
                    ),
                ],
                verbose_feature_names_out=False,
            ).set_output(transform="pandas"),
        ),
        ("to_category", ToCategory(ohe_columns)),
    ]
)

In [156]:
lgbm = LGBMRegressor(random_state=gseed)
fitted_models, oof_preds, metrics, val_scores = cv_result(
    lgbm,
    X_train,
    y_train,
    y_binned,
    rskf,
    tree_pipe_enc,
    "lgbm_regressor_auto_encoder",
)

Оптимизация памяти завершена: 3.20 MB -> 2.93 MB (-8.4%)
[LightGBM] [Info] Auto-choosing col-wise multi-threading, the overhead of testing was 0.001066 seconds.
You can set `force_col_wise=true` to remove the overhead.
[LightGBM] [Info] Total Bins 3308
[LightGBM] [Info] Number of data points in the train set: 1314, number of used features: 75
[LightGBM] [Info] Start training from score 12.024176
Оптимизация памяти завершена: 3.20 MB -> 2.93 MB (-8.4%)
Оптимизация памяти завершена: 0.36 MB -> 0.33 MB (-8.4%)


C:\Users\kochn\AppData\Local\Temp\ipykernel_27260\1431418370.py:17: Pandas4Warning: Constructing a Categorical with a dtype and values containing non-null entries not in that dtype's categories is deprecated and will raise in a future version.
  X_copy[col] = pd.Categorical(X_copy[col], categories=self.categories_[col])


Оптимизация памяти завершена: 3.20 MB -> 2.93 MB (-8.4%)
[LightGBM] [Info] Auto-choosing col-wise multi-threading, the overhead of testing was 0.000645 seconds.
You can set `force_col_wise=true` to remove the overhead.
[LightGBM] [Info] Total Bins 3301
[LightGBM] [Info] Number of data points in the train set: 1314, number of used features: 75
[LightGBM] [Info] Start training from score 12.022495
Оптимизация памяти завершена: 3.20 MB -> 2.93 MB (-8.4%)
Оптимизация памяти завершена: 0.36 MB -> 0.33 MB (-8.4%)


C:\Users\kochn\AppData\Local\Temp\ipykernel_27260\1431418370.py:17: Pandas4Warning: Constructing a Categorical with a dtype and values containing non-null entries not in that dtype's categories is deprecated and will raise in a future version.
  X_copy[col] = pd.Categorical(X_copy[col], categories=self.categories_[col])


Оптимизация памяти завершена: 3.20 MB -> 2.93 MB (-8.4%)
[LightGBM] [Info] Auto-choosing col-wise multi-threading, the overhead of testing was 0.000707 seconds.
You can set `force_col_wise=true` to remove the overhead.
[LightGBM] [Info] Total Bins 3311
[LightGBM] [Info] Number of data points in the train set: 1314, number of used features: 75
[LightGBM] [Info] Start training from score 12.024325
Оптимизация памяти завершена: 3.20 MB -> 2.93 MB (-8.4%)
Оптимизация памяти завершена: 0.36 MB -> 0.33 MB (-8.4%)
Оптимизация памяти завершена: 3.20 MB -> 2.93 MB (-8.4%)
[LightGBM] [Info] Auto-choosing col-wise multi-threading, the overhead of testing was 0.000715 seconds.
You can set `force_col_wise=true` to remove the overhead.
[LightGBM] [Info] Total Bins 3302
[LightGBM] [Info] Number of data points in the train set: 1314, number of used features: 75
[LightGBM] [Info] Start training from score 12.024615
Оптимизация памяти завершена: 3.20 MB -> 2.93 MB (-8.4%)
Оптимизация памяти завершена: 0

...

C:\Users\kochn\AppData\Local\Temp\ipykernel_27260\1431418370.py:17: Pandas4Warning: Constructing a Categorical with a dtype and values containing non-null entries not in that dtype's categories is deprecated and will raise in a future version.
  X_copy[col] = pd.Categorical(X_copy[col], categories=self.categories_[col])


Оптимизация памяти завершена: 3.20 MB -> 2.93 MB (-8.4%)
[LightGBM] [Info] Auto-choosing col-wise multi-threading, the overhead of testing was 0.000614 seconds.
You can set `force_col_wise=true` to remove the overhead.
[LightGBM] [Info] Total Bins 3301
[LightGBM] [Info] Number of data points in the train set: 1314, number of used features: 75
[LightGBM] [Info] Start training from score 12.023901
Оптимизация памяти завершена: 3.20 MB -> 2.93 MB (-8.4%)
Оптимизация памяти завершена: 0.36 MB -> 0.33 MB (-8.4%)


C:\Users\kochn\AppData\Local\Temp\ipykernel_27260\1431418370.py:17: Pandas4Warning: Constructing a Categorical with a dtype and values containing non-null entries not in that dtype's categories is deprecated and will raise in a future version.
  X_copy[col] = pd.Categorical(X_copy[col], categories=self.categories_[col])


Оптимизация памяти завершена: 3.20 MB -> 2.93 MB (-8.4%)
[LightGBM] [Info] Auto-choosing col-wise multi-threading, the overhead of testing was 0.000896 seconds.
You can set `force_col_wise=true` to remove the overhead.
[LightGBM] [Info] Total Bins 3308
[LightGBM] [Info] Number of data points in the train set: 1314, number of used features: 75
[LightGBM] [Info] Start training from score 12.024486
Оптимизация памяти завершена: 3.20 MB -> 2.93 MB (-8.4%)
Оптимизация памяти завершена: 0.36 MB -> 0.33 MB (-8.4%)


C:\Users\kochn\AppData\Local\Temp\ipykernel_27260\1431418370.py:17: Pandas4Warning: Constructing a Categorical with a dtype and values containing non-null entries not in that dtype's categories is deprecated and will raise in a future version.
  X_copy[col] = pd.Categorical(X_copy[col], categories=self.categories_[col])


In [157]:
metrics

,model,TRAIN_rmsle_MEAN,TRAIN_rmsle_STD,VAL_rmsle_MEAN,VAL_rmsle_STD,OOF_rmsle
0,lgbm_regressor_auto_encoder,0.040354,0.000925,0.126781,0.01489,0.123934


Итог:
* baseline - lgbm regressor

    Предобработка данных: простое заполнение пропусков, использование для категориальных колонок встроенный encoder, числовые не трогаются

In [ ]:
pd.read_csv(Path(r"D:\vs_projects\fp_houses\outputs\comparison_table.csv"))

,model,TRAIN_rmsle_MEAN,TRAIN_rmsle_STD,VAL_rmsle_MEAN,VAL_rmsle_STD,OOF_rmsle
0,linear_regression,0.095523,0.002172,0.185530,0.060848,0.193909
1,ridge_regression,0.106203,0.003154,0.140920,0.039436,0.145016
2,knn_regressor,0.141765,0.001968,0.175327,0.018612,0.174046
3,rfr_regressor,0.053529,0.000883,0.142283,0.017513,0.141632
4,lgbm_oe&ohe,0.044258,0.001054,0.129718,0.016017,0.127076
5,lgbm_oe&lgbmt,0.040354,0.000925,0.126781,0.014890,0.123934
6,lgbm_encoder,0.040153,0.000801,0.126993,0.014737,0.124109
